In [1]:
pip install nba_api pandas

  Using cached pandas-3.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached pandas-3.0.2-cp312-cp312-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl (5.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [nba_api]m7/8 [nba_api]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from nba_api.stats.endpoints import scoreboardv3, playbyplayv3
from datetime import datetime, timedelta
import pandas as pd


def get_recent_game_with_pbp():
    for i in range(7):
        date = datetime.now() - timedelta(days=i)
        date_str = date.strftime('%Y-%m-%d')

        sb = scoreboardv3.ScoreboardV3(game_date=date_str)
        games = sb.get_dict()['scoreboard']['games']

        for game in games:
            game_id = game['gameId']

            pbp = playbyplayv3.PlayByPlayV3(game_id=game_id)
            actions = pbp.get_dict().get('game', {}).get('actions', [])

            if len(actions) > 0:
                print(f"Found valid game: {game_id} on {date_str}")
                return game_id, date_str

    raise Exception("No recent game with play-by-play found")


def get_play_by_play(game_id):
    pbp = playbyplayv3.PlayByPlayV3(game_id=game_id)
    actions = pbp.get_dict()['game']['actions']
    return pd.DataFrame(actions)


if __name__ == "__main__":
    game_id, game_date = get_recent_game_with_pbp()

    pbp_df = get_play_by_play(game_id)

    print(pbp_df.head())
    pbp_df.to_csv("play_by_play.csv", index=False)

Found valid game: 0042500111 on 2026-04-19
   actionNumber        clock  period      teamId teamTricode  personId  \
0             2  PT12M00.00S       1           0                     0   
1             4  PT12M00.00S       1  1610612738         BOS   1629674   
2             7  PT11M48.00S       1  1610612755         PHI    202331   
3             8  PT11M47.00S       1  1610612738         BOS   1628369   
4             9  PT11M38.00S       1  1610612755         PHI   1641737   

  playerName playerNameI  xLegacy  yLegacy  ...  scoreHome scoreAway  \
0                               0        0  ...          0         0   
1      Queta    N. Queta        0        0  ...                        
2     George   P. George      123      234  ...                        
3      Tatum    J. Tatum        0        0  ...                        
4       Bona     A. Bona        0        0  ...                        

   pointsTotal location                                 description  \
0       

In [5]:
from nba_api.stats.endpoints import scoreboardv3, playbyplayv3
from datetime import datetime, timedelta
import pandas as pd


def get_recent_game_with_pbp():
    for i in range(7):
        date = datetime.now() - timedelta(days=i)
        date_str = date.strftime('%Y-%m-%d')

        sb = scoreboardv3.ScoreboardV3(game_date=date_str)
        games = sb.get_dict()['scoreboard']['games']

        for game in games:
            game_id = game['gameId']

            pbp = playbyplayv3.PlayByPlayV3(game_id=game_id)
            actions = pbp.get_dict().get('game', {}).get('actions', [])

            if len(actions) > 0:
                print(f"Found valid game: {game_id} on {date_str}")
                return game_id

    raise Exception("No recent game with play-by-play found")


def get_play_by_play(game_id):
    pbp = playbyplayv3.PlayByPlayV3(game_id=game_id)
    actions = pbp.get_dict()['game']['actions']
    return pd.DataFrame(actions)


def enrich_pbp(df):
    # --- SCORE HANDLING ---
    df['scoreHome'] = pd.to_numeric(df['scoreHome'], errors='coerce')
    df['scoreAway'] = pd.to_numeric(df['scoreAway'], errors='coerce')

    # forward fill missing scores
    df['scoreHome'] = df['scoreHome'].ffill().fillna(0)
    df['scoreAway'] = df['scoreAway'].ffill().fillna(0)

    # score after play
    df['score_after'] = df['scoreAway'].astype(int).astype(str) + "-" + df['scoreHome'].astype(int).astype(str)

    # score before play
    df['score_before'] = df['score_after'].shift(1).fillna("0-0")

    # --- SCORE DIFFERENTIAL ---
    df['diff'] = df['scoreHome'] - df['scoreAway']

    # --- LEAD CHANGE DETECTION ---
    df['leader'] = df['diff'].apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
    df['lead_change'] = df['leader'].diff().fillna(0) != 0

    # --- RUN DETECTION ---
    df['home_pts'] = df['scoreHome'].diff().fillna(0)
    df['away_pts'] = df['scoreAway'].diff().fillna(0)

    df['run_team'] = df.apply(
        lambda row: 'HOME' if row['home_pts'] > 0 else ('AWAY' if row['away_pts'] > 0 else None),
        axis=1
    )

    # track runs
    runs = []
    current_team = None
    current_points = 0

    for _, row in df.iterrows():
        if row['run_team'] == current_team:
            current_points += max(row['home_pts'], row['away_pts'])
        else:
            current_team = row['run_team']
            current_points = max(row['home_pts'], row['away_pts'])

        runs.append(current_points)

    df['current_run'] = runs

    return df


if __name__ == "__main__":
    game_id = get_recent_game_with_pbp()
    pbp_df = get_play_by_play(game_id)

    pbp_df = enrich_pbp(pbp_df)

    print(pbp_df[['period', 'clock', 'description', 'score_before', 'score_after', 'current_run']].head(20))

    pbp_df.to_csv("play_by_play_enriched.csv", index=False)

Found valid game: 0042500111 on 2026-04-19
    period        clock                                       description  \
0        1  PT12M00.00S                 Start of 1st Period (1:11 PM EST)   
1        1  PT12M00.00S        Jump Ball Queta vs. Bona: Tip to Edgecombe   
2        1  PT11M48.00S                     MISS George 26' 3PT Jump Shot   
3        1  PT11M47.00S                       Tatum REBOUND (Off:0 Def:1)   
4        1  PT11M38.00S                    Bona S.FOUL (P1.T1) (S.Foster)   
5        1  PT11M38.00S                   Queta Free Throw 1 of 2 (1 PTS)   
6        1  PT11M38.00S                      MISS Queta Free Throw 2 of 2   
7        1  PT11M38.00S                   Oubre Jr. REBOUND (Off:0 Def:1)   
8        1  PT11M24.00S                      MISS Edgecombe 3PT Jump Shot   
9        1  PT11M23.00S                       Tatum REBOUND (Off:0 Def:2)   
10       1  PT11M15.00S    Hauser 26' 3PT Jump Shot (3 PTS) (Brown 1 AST)   
11       1  PT10M57.00S    Maxey 

In [8]:
import json
import sys
from pathlib import Path

from nba_api.stats.endpoints import playbyplayv3

# Resolve repo root when the kernel cwd is notebooks/ or project root.
_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "requirements.txt").is_file() and (_p / "notebooks").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from src.utils.paths import repo_root

# Set this to your fixture game (or reuse game_id from an earlier cell).
game_id = "0042500111"

out_dir = repo_root() / "data" / "raw" / "pbp"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{game_id}.json"

FORCE_REFRESH = False

if out_path.exists() and not FORCE_REFRESH:
    print(f"Using cached file (set FORCE_REFRESH=True to re-fetch): {out_path}")
else:
    pbp = playbyplayv3.PlayByPlayV3(game_id=game_id)
    payload = pbp.get_dict()
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"Wrote {out_path} ({out_path.stat().st_size / 1e6:.2f} MB)")

if out_path.exists():
    print(f"On disk: {out_path} ({out_path.stat().st_size / 1e6:.2f} MB)")

Using cached file (set FORCE_REFRESH=True to re-fetch): /Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0042500111.json
On disk: /Users/manishkavuri/Desktop/nba-lineup-decision-engine/data/raw/pbp/0042500111.json (0.33 MB)


In [10]:
from pathlib import Path
import json
import pandas as pd
import sys

_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "requirements.txt").is_file() and (_p / "notebooks").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

from src.utils.paths import repo_root

game_id = "0042500111"  # e.g. game_id from scoreboard

# expected raw file path
raw_path = repo_root() / "data" / "raw" / "pbp" / f"{game_id}.json"

#load raw payload
with raw_path.open("r", encoding="utf-8") as f:
    payload = json.load(f)

actions = payload.get('game', {}).get('actions', [])
assert len(actions) > 0, "No actions found in payload['game']['actions']"

# convert to dataframe
pbp_df = pd.DataFrame(payload['game']['actions'])

# sort by actionNumber if present
if 'actionNumber' in pbp_df.columns:
    pbp_df = pbp_df.sort_values('actionNumber')

# reset index
pbp_df = pbp_df.reset_index(drop=True)

print(f"Loaded {len(pbp_df):,} actions")
print(f"Shape: {pbp_df.shape}")

# columnn contract summary
print("\n=== COLUMNS ===")
print(pbp_df.columns.tolist())

print("\n=== DTYPES ===")
print(pbp_df.dtypes)

print("\n=== NULL RATES ===")
null_rates = (pbp_df.isna().mean() * 100).sort_values(ascending=False).round(2)
print(null_rates)

# inspect actionType values
if 'actionType' in pbp_df.columns:
    print("\n=== ACTION TYPE VALUES ===")
    print(pbp_df['actionType'].value_counts())

# find substitution-like rows
# start with actionType
sub_mask = pd.Series(False, index=pbp_df.index)

text_cols = [c for c in ["actionType", "description", "subType", "clock"] if c in pbp_df.columns]

for col in text_cols:
    sub_mask |= pbp_df[col].str.contains("sub", case=False, na=False)

# find rows where subType is "In" or "Out"
sub_rows = pbp_df.loc[sub_mask].copy()

print(f"\n=== SUBSTITUTION-LIKE ROW COUNT ===\n{sub_rows.shape[0]}")

# show a compact preview first
preview_cols = [c for c in [
    "actionNumber", "period", "clock", "teamId", "personId",
    "actionType", "subType", "description"
] if c in sub_rows.columns]

if len(sub_rows) > 0:
    print("\n=== SUBSTITUTION-LIKE ROW PREVIEW ===")
    display(sub_rows[preview_cols].head(10))

    print("\n=== ONE FULL SUBSTITUTION-LIKE ROW AS DICT ===")
    sample_row = sub_rows.iloc[0].to_dict()
    for k, v in sample_row.items():
        print(f"{k}: {v}")
else:
    print("No substitution-like rows found with the current filter.")

Loaded 485 actions
Shape: (485, 23)

=== COLUMNS ===
['actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId']

=== DTYPES ===
actionNumber      int64
clock               str
period            int64
teamId            int64
teamTricode         str
personId          int64
playerName          str
playerNameI         str
xLegacy           int64
yLegacy           int64
shotDistance      int64
shotResult          str
isFieldGoal       int64
scoreHome           str
scoreAway           str
pointsTotal       int64
location            str
description         str
actionType          str
subType             str
videoAvailable    int64
shotValue         int64
actionId          int64
dtype: object

=== NULL RATES ===
actionNumber      0.0
isFieldGoal       

,actionNumber,period,clock,teamId,personId,actionType,subType,description
16,24,1,PT10M23.00S,1610612755,1641737,Substitution,,SUB: Drummond FOR Bona
34,49,1,PT08M39.00S,1610612738,1629674,Substitution,,SUB: Vucevic FOR Queta
67,91,1,PT04M50.00S,1610612738,1630573,Substitution,,SUB: Pritchard FOR Hauser
68,92,1,PT04M50.00S,1610612755,1626162,Substitution,,SUB: Grimes FOR Oubre Jr.
69,93,1,PT04M50.00S,1610612755,1642845,Substitution,,SUB: Barlow FOR Edgecombe
74,106,1,PT04M39.00S,1610612755,203083,Substitution,,SUB: Oubre Jr. FOR Drummond
85,120,1,PT03M49.00S,1610612738,1627759,Substitution,,SUB: Walsh FOR Brown
96,133,1,PT02M39.00S,1610612755,202331,Substitution,,SUB: Edwards FOR George
114,157,1,PT00M27.30S,1610612738,202696,Substitution,,SUB: Brown FOR Vucevic
115,158,1,PT00M27.30S,1610612755,1631230,Substitution,,SUB: George FOR Barlow



=== ONE FULL SUBSTITUTION-LIKE ROW AS DICT ===
actionNumber: 24
clock: PT10M23.00S
period: 1
teamId: 1610612755
teamTricode: PHI
personId: 1641737
playerName: Bona
playerNameI: A. Bona
xLegacy: 0
yLegacy: 0
shotDistance: 0
shotResult: 
isFieldGoal: 0
scoreHome: 
scoreAway: 
pointsTotal: 0
location: v
description: SUB: Drummond FOR Bona
actionType: Substitution
subType: 
videoAvailable: 0
shotValue: 0
actionId: 17


In [11]:
import re
from typing import Optional

_CLOCK_RE = re.compile(
    r"^PT"
    r"(?:(?P<hours>\d+)H)?"
    r"(?:(?P<minutes>\d+)M)?"
    r"(?:(?P<seconds>\d+(?:\.\d+)?)S)?"
    r"$",
    re.IGNORECASE,
)


def parse_clock_seconds_remaining(clock: Optional[object]) -> float:
    """Seconds remaining in the current period (NBA PlayByPlay `clock`)."""
    if clock is None:
        return float("nan")
    s = str(clock).strip()
    if not s:
        return float("nan")

    m = _CLOCK_RE.match(s)
    if not m:
        raise ValueError(f"Unparseable clock: {clock!r}")

    hours = float(m.group("hours") or 0)
    minutes = float(m.group("minutes") or 0)
    seconds = float(m.group("seconds") or 0)
    return hours * 3600 + minutes * 60 + seconds

pbp_df["sec_left_period"] = pbp_df["clock"].map(parse_clock_seconds_remaining)
g = pbp_df.sort_values("actionNumber").groupby("period")["sec_left_period"]
violations = (g.diff() > 1e-6)  # clock should not go UP within a period
print("rows where clock increased:", violations.sum())

rows where clock increased: 4


In [12]:
import pandas as pd

# Ensure chronological order within the feed
df = pbp_df.sort_values(["period", "actionNumber"], kind="mergesort").copy()

df["sec_left_prev"] = df.groupby("period")["sec_left_period"].shift(1)
df["clock_jump"] = df["sec_left_period"] - df["sec_left_prev"]  # should be <= 0 if monotone

# tolerate tiny float noise (optional)
EPS = 1e-6
viol = df[df["clock_jump"] > EPS].copy()

print(f"violations: {len(viol)}")
display(
    viol[
        [
            "actionNumber",
            "period",
            "clock",
            "sec_left_period",
            "sec_left_prev",
            "clock_jump",
            "actionType",
            "description",
        ]
    ]
)

idx = set(viol.index)
ctx_idx = set()
for i in viol.index:
    loc = df.index.get_loc(i)
    if isinstance(loc, slice):
        # unlikely for unique index
        continue
    for j in (loc - 1, loc, loc + 1):
        if 0 <= j < len(df):
            ctx_idx.add(df.index[j])

ctx = df.loc[sorted(ctx_idx)].sort_values(["period", "actionNumber"])

display(
    ctx[
        [
            "actionNumber",
            "period",
            "clock",
            "sec_left_period",
            "actionType",
            "subType",
            "teamId",
            "personId",
            "scoreHome",
            "scoreAway",
            "description",
        ]
    ]
)

violations: 4


,actionNumber,period,clock,sec_left_period,sec_left_prev,clock_jump,actionType,description
39,56,1,PT08M48.00S,528.0,490.0,38.0,Turnover,Brown Lost Ball Turnover (P1.T2)
207,289,2,PT05M28.00S,328.0,277.0,51.0,Missed Shot,MISS Edwards 3' Tip Layup Shot
246,344,2,PT01M50.00S,110.0,100.0,10.0,Substitution,SUB: Brown FOR Pritchard
316,439,2,PT03M17.00S,197.0,0.0,197.0,Turnover,Vucevic Lost Ball Turnover (P2.T5)


,actionNumber,period,clock,sec_left_period,actionType,subType,teamId,personId,scoreHome,scoreAway,description
38,55,1,PT08M10.00S,490.0,Rebound,Unknown,1610612738,1630573,,,Hauser REBOUND (Off:0 Def:1)
39,56,1,PT08M48.00S,528.0,Turnover,Lost Ball,1610612738,1627759,,,Brown Lost Ball Turnover (P1.T2)
40,56,1,PT08M48.00S,528.0,,,1610612755,203083,,,Drummond STEAL (1 STL)
206,288,2,PT04M37.00S,277.0,Rebound,Unknown,1610612738,202696,,,Vucevic REBOUND (Off:0 Def:3)
207,289,2,PT05M28.00S,328.0,Missed Shot,Tip Layup Shot,1610612755,1642348,,,MISS Edwards 3' Tip Layup Shot
208,290,2,PT05M27.00S,327.0,Rebound,Unknown,1610612755,1642348,,,Edwards REBOUND (Off:3 Def:2)
245,343,2,PT01M40.00S,100.0,Turnover,Offensive Foul Turnover,1610612738,1627759,,,Brown Offensive Foul Turnover (P2.T6)
246,344,2,PT01M50.00S,110.0,Substitution,,1610612738,1630202,,,SUB: Brown FOR Pritchard
247,346,2,PT01M40.00S,100.0,Substitution,,1610612738,1627759,,,SUB: Pritchard FOR Brown
265,370,2,PT00M00.00S,0.0,period,end,0,0,64,46,End of 2nd Period (2:19 PM EST)


In [ ]:
import re
from typing import Optional

import pandas as pd
from nba_api.stats.endpoints import boxscoretraditionalv2

# --- (2) Opening lineups from box score (starters = non-empty START_POSITION) ---

def load_player_stats(game_id: str) -> pd.DataFrame:
    box = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id)
    return box.player_stats.get_data_frame()


def opening_lineups_by_team_id(game_id: str) -> dict[int, set[int]]:
    """
    Returns {team_id: {5 starter player_ids}} for this game.
    """
    players = load_player_stats(game_id)
    if "START_POSITION" not in players.columns:
        raise KeyError("Expected START_POSITION on player_stats; check nba_api version.")

    starters = players[players["START_POSITION"].fillna("") != ""].copy()
    out: dict[int, set[int]] = {}
    for tid, grp in starters.groupby("TEAM_ID"):
        ids = set(grp["PLAYER_ID"].astype(int).tolist())
        out[int(tid)] = ids
        if len(ids) != 5:
            print(f"Warning: team {tid} has {len(ids)} starters (expected 5).")
    if len(out) != 2:
        print(f"Warning: expected 2 teams with starters, got {len(out)}.")
    return out


# --- (1) Substitution description parsing + name -> PLAYER_ID ---

_SUB_RE = re.compile(r"^SUB:\s*(.+?)\s+FOR\s+(.+?)\s*$", re.IGNORECASE)


def parse_sub_description(description: str) -> Optional[tuple[str, str]]:
    if not isinstance(description, str):
        return None
    m = _SUB_RE.match(description.strip())
    if not m:
        return None
    return m.group(1).strip(), m.group(2).strip()


def _norm_tokens(s: str) -> list[str]:
    s = s.lower().replace(".", " ")
    parts = [p for p in re.split(r"\s+", s.strip()) if p]
    junk = {"jr", "sr", "ii", "iii", "iv", "v"}
    return [p for p in parts if p not in junk]


def resolve_name_to_player_id(name: str, team_id: int, players: pd.DataFrame) -> int:
    """
    Map a PBP sub name (often 'Drummond', 'Oubre Jr.') to PLAYER_ID within that team.
    """
    name = str(name).strip()
    team_players = players[players["TEAM_ID"] == int(team_id)]
    if team_players.empty:
        raise ValueError(f"No players for team_id={team_id}")

    # Exact match on full box score name
    exact = team_players[team_players["PLAYER_NAME"].str.lower() == name.lower()]
    if len(exact) == 1:
        return int(exact.iloc[0]["PLAYER_ID"])

    # Match on last "real" token (handles 'Drummond', 'Oubre Jr.')
    toks = _norm_tokens(name)
    last = toks[-1] if toks else name.lower()

    # Last-name match against last token of PLAYER_NAME
    last_name_match = team_players[
        team_players["PLAYER_NAME"].str.lower().str.split().str[-1] == last
    ]
    if len(last_name_match) == 1:
        return int(last_name_match.iloc[0]["PLAYER_ID"])

    # Substring match (handles nicknames / unusual strings)
    sub = team_players[team_players["PLAYER_NAME"].str.lower().str.contains(re.escape(last), na=False)]
    if len(sub) == 1:
        return int(sub.iloc[0]["PLAYER_ID"])

    raise ValueError(
        f"Ambiguous/unresolved name {name!r} for team {team_id}. "
        f"Candidates (last-name): {len(last_name_match)}, (contains): {len(sub)}"
    )


def substitution_player_ids(row: pd.Series, players: pd.DataFrame) -> Optional[tuple[int, int, int]]:
    """
    For a substitution PBP row, return (team_id, player_in_id, player_out_id) or None.
    """
    if str(row.get("actionType", "")).strip().lower() != "substitution":
        return None
    desc = row.get("description")
    parsed = parse_sub_description(desc) if isinstance(desc, str) else None
    if not parsed:
        return None

    player_in_name, player_out_name = parsed
    tid = int(row["teamId"])
    pid_in = resolve_name_to_player_id(player_in_name, tid, players)
    pid_out = resolve_name_to_player_id(player_out_name_api, tid, players)
    return tid, pid_in, pid_out


# --- Example wiring for your cached game ---

game_id = "0042500111"

players_df = load_player_stats(game_id)
starters_by_team = opening_lineups_by_team_id(game_id)
print("Starters by team_id:", {k: sorted(v) for k, v in starters_by_team.items()})

subs = pbp_df[pbp_df["actionType"].astype(str).str.lower() == "substitution"].copy()
parsed_rows = []
for _, r in subs.iterrows():
    try:
        triplet = substitution_player_ids(r, players_df)
        parsed_rows.append(
            {
                "actionNumber": r.get("actionNumber"),
                "clock": r.get("clock"),
                "teamId": r.get("teamId"),
                "description": r.get("description"),
                "triplet": triplet,
            }
        )
    except Exception as e:
        parsed_rows.append(
            {
                "actionNumber": r.get("actionNumber"),
                "clock": r.get("clock"),
                "teamId": r.get("teamId"),
                "description": r.get("description"),
                "error": repr(e),
            }
        )

display(pd.DataFrame(parsed_rows).head(15))

/var/folders/m4/30h3h5c94fvfdkd1zs59v6w80000gn/T/ipykernel_45654/3453960051.py:10: DeprecationWarning: BoxScoreTraditionalV2 is deprecated and will be removed in a future version. Please use BoxScoreTraditionalV3 instead. Data is no longer being published for BoxScoreTraditionalV2 as of the 2025-26 NBA season.
  box = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id)


Starters by team_id: {1610612738: [1627759, 1628369, 1628401, 1629674, 1630573], 1610612755: [202331, 1626162, 1630178, 1641737, 1642845]}


,actionNumber,clock,teamId,description,triplet,error
0,24,PT10M23.00S,1610612755,SUB: Drummond FOR Bona,"(1610612755, 203083, 1641737)",NaN
1,49,PT08M39.00S,1610612738,SUB: Vucevic FOR Queta,NaN,"ValueError(""Ambiguous/unresolved name 'Vucevic..."
2,91,PT04M50.00S,1610612738,SUB: Pritchard FOR Hauser,"(1610612738, 1630202, 1630573)",NaN
3,92,PT04M50.00S,1610612755,SUB: Grimes FOR Oubre Jr.,"(1610612755, 1629656, 1626162)",NaN
4,93,PT04M50.00S,1610612755,SUB: Barlow FOR Edgecombe,"(1610612755, 1631230, 1642845)",NaN
5,106,PT04M39.00S,1610612755,SUB: Oubre Jr. FOR Drummond,"(1610612755, 1626162, 203083)",NaN
6,120,PT03M49.00S,1610612738,SUB: Walsh FOR Brown,"(1610612738, 1641775, 1627759)",NaN
7,133,PT02M39.00S,1610612755,SUB: Edwards FOR George,"(1610612755, 1642348, 202331)",NaN
8,157,PT00M27.30S,1610612738,SUB: Brown FOR Vucevic,NaN,"ValueError(""Ambiguous/unresolved name 'Vucevic..."
9,158,PT00M27.30S,1610612755,SUB: George FOR Barlow,"(1610612755, 202331, 1631230)",NaN


In [14]:
from src.processing.lineup_stints import build_lineup_stints_for_game

stints, warnings = build_lineup_stints_for_game("0042500111")

In [15]:
print(stints)

       game_id  period  start_clock    end_clock  duration_seconds  \
0   0042500111       1  PT12M00.00S  PT10M23.00S              97.0   
1   0042500111       1  PT10M23.00S  PT08M39.00S             104.0   
2   0042500111       1  PT08M39.00S  PT04M50.00S             229.0   
3   0042500111       1  PT04M50.00S  PT04M50.00S               0.0   
4   0042500111       1  PT04M50.00S  PT04M50.00S               0.0   
5   0042500111       1  PT04M50.00S  PT04M39.00S              11.0   
6   0042500111       1  PT04M39.00S  PT03M49.00S              50.0   
7   0042500111       1  PT03M49.00S  PT02M39.00S              70.0   
8   0042500111       1  PT02M39.00S  PT00M27.30S             131.7   
9   0042500111       1  PT00M27.30S  PT00M27.30S               0.0   
10  0042500111       1  PT00M27.30S  PT00M00.00S              27.3   
11  0042500111       2  PT12M00.00S  PT02M47.00S             553.0   
12  0042500111       2  PT02M47.00S  PT01M50.00S              57.0   
13  0042500111      

In [16]:
import pandas as pd

from src.processing.lineup_stints import build_lineup_stints_for_game

game_id = "0042500111"

stints, warnings = build_lineup_stints_for_game(game_id)

print(f"stints rows: {len(stints):,}")
print(f"warnings: {len(warnings):,}")
if warnings:
    print("\nFirst 10 warnings:")
    for w in warnings[:10]:
        print(" -", w)

required_cols = [
    "game_id", "period", "start_clock", "end_clock", "duration_seconds",
    "home_team_id", "away_team_id", "home_player_ids", "away_player_ids",
    "score_home_start", "score_away_start", "score_home_end", "score_away_end",
    "ended_by_substitution",
]
missing = [c for c in required_cols if c not in stints.columns]
assert not missing, f"Missing stint columns: {missing}"

# --- Basic structural checks ---
assert (stints["game_id"] == game_id).all(), "Unexpected game_id values in stints"
assert stints["period"].notna().all(), "Null period values"
assert stints["duration_seconds"].notna().all(), "Null durations"

# parse lineup csv -> lists
home_lists = stints["home_player_ids"].fillna("").str.split(",")
away_lists = stints["away_player_ids"].fillna("").str.split(",")

# Every row should have 5+5 players (if your current implementation skips bad subs, this may still pass)
bad_home_n = home_lists.map(len) != 5
bad_away_n = away_lists.map(len) != 5

# No duplicates inside each lineup
dup_home = home_lists.map(lambda xs: len(xs) != len(set(xs)))
dup_away = away_lists.map(lambda xs: len(xs) != len(set(xs)))

print("\n=== Cardinality Checks ===")
print("home lineup != 5:", int(bad_home_n.sum()))
print("away lineup != 5:", int(bad_away_n.sum()))
print("home lineup duplicates:", int(dup_home.sum()))
print("away lineup duplicates:", int(dup_away.sum()))

# Durations
neg_dur = stints["duration_seconds"] < 0
zero_dur = stints["duration_seconds"] == 0

print("\n=== Duration Checks ===")
print("negative durations:", int(neg_dur.sum()))
print("zero durations:", int(zero_dur.sum()))
print("total stint seconds:", round(float(stints["duration_seconds"].sum()), 2))

# Score consistency
bad_score = (
    (stints["score_home_end"] < stints["score_home_start"])
    | (stints["score_away_end"] < stints["score_away_start"])
)
print("\n=== Score Monotonicity ===")
print("rows where score decreased:", int(bad_score.sum()))

# Period-level summary
summary = (
    stints.groupby("period", as_index=False)
    .agg(
        n_stints=("period", "size"),
        total_seconds=("duration_seconds", "sum"),
        ended_by_sub=("ended_by_substitution", "sum"),
    )
    .sort_values("period")
)
print("\n=== Period Summary ===")
display(summary)

# Show problematic rows if any
problem_mask = bad_home_n | bad_away_n | dup_home | dup_away | neg_dur | bad_score
if problem_mask.any():
    print("\n=== Problem Rows (first 20) ===")
    display(
        stints.loc[problem_mask, [
            "period", "start_clock", "end_clock", "duration_seconds",
            "home_player_ids", "away_player_ids",
            "score_home_start", "score_home_end", "score_away_start", "score_away_end",
            "ended_by_substitution",
        ]].head(20)
    )
else:
    print("\nNo structural problems found in this QA cell.")

print("\n=== Sample Stints ===")
display(stints.head(15))

stints rows: 29
warnings: 21

First 10 warnings:
 - actionNumber=221: player 1630568 not on floor for team 1610612738, have {1628401, 1628369, 1627759, 1630202, 1641775}
 - actionNumber=222: player 1631248 not on floor for team 1610612738, have {1628401, 1628369, 1627759, 1630202, 1641775}
 - actionNumber=223: player 1630178 already on floor for team 1610612755
 - actionNumber=241: player 1630573 not on floor for team 1610612738, have {1628401, 1628369, 1627759, 1630202, 1641775}
 - actionNumber=242: player 203083 not on floor for team 1610612755, have {1630178, 1642348, 1626162, 1629656, 202331}
 - actionNumber=250: player 1626162 already on floor for team 1610612755
 - actionNumber=274: player 202331 already on floor for team 1610612755
 - actionNumber=285: player 1629674 not on floor for team 1610612738, have {1628401, 1628369, 1627759, 1630202, 1641775}
 - actionNumber=330: player 202696 not on floor for team 1610612738, have {1628401, 1628369, 1630202, 1630573, 1641775}
 - actionN

,period,n_stints,total_seconds,ended_by_sub
0,1,11,720.0,10
1,2,5,1440.0,3
2,3,3,339.0,2
3,4,10,720.0,9



No structural problems found in this QA cell.

=== Sample Stints ===


,game_id,period,start_clock,end_clock,duration_seconds,home_team_id,away_team_id,home_player_ids,away_player_ids,score_home_start,score_away_start,score_home_end,score_away_end,ended_by_substitution
0,0042500111,1,PT12M00.00S,PT10M23.00S,97.0,1610612738,1610612755,"1627759,1628369,1628401,1629674,1630573","202331,1626162,1630178,1641737,1642845",0,0,4,3,True
1,0042500111,1,PT10M23.00S,PT08M39.00S,104.0,1610612738,1610612755,"1627759,1628369,1628401,1629674,1630573","202331,203083,1626162,1630178,1642845",4,3,8,6,True
2,0042500111,1,PT08M39.00S,PT04M50.00S,229.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630573","202331,203083,1626162,1630178,1642845",8,6,22,11,True
3,0042500111,1,PT04M50.00S,PT04M50.00S,0.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630202","202331,203083,1626162,1630178,1642845",22,11,22,11,True
4,0042500111,1,PT04M50.00S,PT04M50.00S,0.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630202","202331,203083,1629656,1630178,1642845",22,11,22,11,True
5,0042500111,1,PT04M50.00S,PT04M39.00S,11.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630202","202331,203083,1629656,1630178,1631230",22,11,22,11,True
6,0042500111,1,PT04M39.00S,PT03M49.00S,50.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630202","202331,1626162,1629656,1630178,1631230",22,11,22,13,True
7,0042500111,1,PT03M49.00S,PT02M39.00S,70.0,1610612738,1610612755,"202696,1628369,1628401,1630202,1641775","202331,1626162,1629656,1630178,1631230",22,13,24,14,True
8,0042500111,1,PT02M39.00S,PT00M27.30S,131.7,1610612738,1610612755,"202696,1628369,1628401,1630202,1641775","1626162,1629656,1630178,1631230,1642348",24,14,31,18,True
9,0042500111,1,PT00M27.30S,PT00M27.30S,0.0,1610612738,1610612755,"1627759,1628369,1628401,1630202,1641775","1626162,1629656,1630178,1631230,1642348",31,18,31,18,True


In [17]:
import pandas as pd

from src.processing.lineup_stints import build_lineup_stints_for_game

game_id = "0042500111"

stints, warnings = build_lineup_stints_for_game(game_id)

print(f"stints rows: {len(stints):,}")
print(f"warnings: {len(warnings):,}")
if warnings:
    print("\nFirst 10 warnings:")
    for w in warnings[:10]:
        print(" -", w)

required_cols = [
    "game_id", "period", "start_clock", "end_clock", "duration_seconds",
    "home_team_id", "away_team_id", "home_player_ids", "away_player_ids",
    "score_home_start", "score_away_start", "score_home_end", "score_away_end",
    "ended_by_substitution",
]
missing = [c for c in required_cols if c not in stints.columns]
assert not missing, f"Missing stint columns: {missing}"

# --- Basic structural checks ---
assert (stints["game_id"] == game_id).all(), "Unexpected game_id values in stints"
assert stints["period"].notna().all(), "Null period values"
assert stints["duration_seconds"].notna().all(), "Null durations"

# parse lineup csv -> lists
home_lists = stints["home_player_ids"].fillna("").str.split(",")
away_lists = stints["away_player_ids"].fillna("").str.split(",")

# Every row should have 5+5 players (if your current implementation skips bad subs, this may still pass)
bad_home_n = home_lists.map(len) != 5
bad_away_n = away_lists.map(len) != 5

# No duplicates inside each lineup
dup_home = home_lists.map(lambda xs: len(xs) != len(set(xs)))
dup_away = away_lists.map(lambda xs: len(xs) != len(set(xs)))

print("\n=== Cardinality Checks ===")
print("home lineup != 5:", int(bad_home_n.sum()))
print("away lineup != 5:", int(bad_away_n.sum()))
print("home lineup duplicates:", int(dup_home.sum()))
print("away lineup duplicates:", int(dup_away.sum()))

# Durations
neg_dur = stints["duration_seconds"] < 0
zero_dur = stints["duration_seconds"] == 0

print("\n=== Duration Checks ===")
print("negative durations:", int(neg_dur.sum()))
print("zero durations:", int(zero_dur.sum()))
print("total stint seconds:", round(float(stints["duration_seconds"].sum()), 2))

# Score consistency
bad_score = (
    (stints["score_home_end"] < stints["score_home_start"])
    | (stints["score_away_end"] < stints["score_away_start"])
)
print("\n=== Score Monotonicity ===")
print("rows where score decreased:", int(bad_score.sum()))

# Period-level summary
summary = (
    stints.groupby("period", as_index=False)
    .agg(
        n_stints=("period", "size"),
        total_seconds=("duration_seconds", "sum"),
        ended_by_sub=("ended_by_substitution", "sum"),
    )
    .sort_values("period")
)
print("\n=== Period Summary ===")
display(summary)

# Show problematic rows if any
problem_mask = bad_home_n | bad_away_n | dup_home | dup_away | neg_dur | bad_score
if problem_mask.any():
    print("\n=== Problem Rows (first 20) ===")
    display(
        stints.loc[problem_mask, [
            "period", "start_clock", "end_clock", "duration_seconds",
            "home_player_ids", "away_player_ids",
            "score_home_start", "score_home_end", "score_away_start", "score_away_end",
            "ended_by_substitution",
        ]].head(20)
    )
else:
    print("\nNo structural problems found in this QA cell.")

print("\n=== Sample Stints ===")
display(stints.head(15))

stints rows: 29
warnings: 21

First 10 warnings:
 - actionNumber=221: player 1630568 not on floor for team 1610612738, have {1628401, 1628369, 1627759, 1630202, 1641775}
 - actionNumber=222: player 1631248 not on floor for team 1610612738, have {1628401, 1628369, 1627759, 1630202, 1641775}
 - actionNumber=223: player 1630178 already on floor for team 1610612755
 - actionNumber=241: player 1630573 not on floor for team 1610612738, have {1628401, 1628369, 1627759, 1630202, 1641775}
 - actionNumber=242: player 203083 not on floor for team 1610612755, have {1630178, 1642348, 1626162, 1629656, 202331}
 - actionNumber=250: player 1626162 already on floor for team 1610612755
 - actionNumber=274: player 202331 already on floor for team 1610612755
 - actionNumber=285: player 1629674 not on floor for team 1610612738, have {1628401, 1628369, 1627759, 1630202, 1641775}
 - actionNumber=330: player 202696 not on floor for team 1610612738, have {1628401, 1628369, 1630202, 1630573, 1641775}
 - actionN

,period,n_stints,total_seconds,ended_by_sub
0,1,11,720.0,10
1,2,5,1440.0,3
2,3,3,339.0,2
3,4,10,720.0,9



No structural problems found in this QA cell.

=== Sample Stints ===


,game_id,period,start_clock,end_clock,duration_seconds,home_team_id,away_team_id,home_player_ids,away_player_ids,score_home_start,score_away_start,score_home_end,score_away_end,ended_by_substitution
0,0042500111,1,PT12M00.00S,PT10M23.00S,97.0,1610612738,1610612755,"1627759,1628369,1628401,1629674,1630573","202331,1626162,1630178,1641737,1642845",0,0,4,3,True
1,0042500111,1,PT10M23.00S,PT08M39.00S,104.0,1610612738,1610612755,"1627759,1628369,1628401,1629674,1630573","202331,203083,1626162,1630178,1642845",4,3,8,6,True
2,0042500111,1,PT08M39.00S,PT04M50.00S,229.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630573","202331,203083,1626162,1630178,1642845",8,6,22,11,True
3,0042500111,1,PT04M50.00S,PT04M50.00S,0.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630202","202331,203083,1626162,1630178,1642845",22,11,22,11,True
4,0042500111,1,PT04M50.00S,PT04M50.00S,0.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630202","202331,203083,1629656,1630178,1642845",22,11,22,11,True
5,0042500111,1,PT04M50.00S,PT04M39.00S,11.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630202","202331,203083,1629656,1630178,1631230",22,11,22,11,True
6,0042500111,1,PT04M39.00S,PT03M49.00S,50.0,1610612738,1610612755,"202696,1627759,1628369,1628401,1630202","202331,1626162,1629656,1630178,1631230",22,11,22,13,True
7,0042500111,1,PT03M49.00S,PT02M39.00S,70.0,1610612738,1610612755,"202696,1628369,1628401,1630202,1641775","202331,1626162,1629656,1630178,1631230",22,13,24,14,True
8,0042500111,1,PT02M39.00S,PT00M27.30S,131.7,1610612738,1610612755,"202696,1628369,1628401,1630202,1641775","1626162,1629656,1630178,1631230,1642348",24,14,31,18,True
9,0042500111,1,PT00M27.30S,PT00M27.30S,0.0,1610612738,1610612755,"1627759,1628369,1628401,1630202,1641775","1626162,1629656,1630178,1631230,1642348",31,18,31,18,True


In [18]:
# Inspect one problematic stoppage
period_target = 2
clock_target = "PT04M50.00S"   # change as needed

view_cols = [
    "actionNumber", "actionId", "period", "clock",
    "teamId", "personId", "actionType", "subType", "description"
]

tmp = pbp_df.sort_values(["period", "actionNumber"], kind="mergesort").copy()
slice_df = tmp[(tmp["period"] == period_target) & (tmp["clock"] == clock_target)]

print(f"rows at ({period_target}, {clock_target}): {len(slice_df)}")
display(slice_df[view_cols])

subs = slice_df[slice_df["actionType"].astype(str).str.lower() == "substitution"]
print(f"sub rows at this clock: {len(subs)}")
display(subs[view_cols])

rows at (2, PT04M50.00S): 1


,actionNumber,actionId,period,clock,teamId,personId,actionType,subType,description
201,281,204,2,PT04M50.00S,1610612755,1630178,Rebound,Unknown,Maxey REBOUND (Off:0 Def:1)


sub rows at this clock: 0


,actionNumber,actionId,period,clock,teamId,personId,actionType,subType,description


In [20]:
# Inspect 7 raw rows around period=1, clock=PT04M50.00S
# Assumes pbp_df already exists and is sorted by actionNumber (if not, we sort below).

import pandas as pd

target_period = 1
target_clock = "PT04M50.00S"

df = pbp_df.copy()
if "actionNumber" in df.columns:
    df = df.sort_values("actionNumber", kind="mergesort").reset_index(drop=True)

mask = (df["period"] == target_period) & (df["clock"] == target_clock)
idxs = df.index[mask].tolist()

print(f"matches for (period={target_period}, clock={target_clock}): {len(idxs)}")

cols = [c for c in [
    "actionNumber", "actionId", "period", "clock",
    "teamId", "teamTricode", "personId", "playerName",
    "actionType", "subType", "description", "scoreHome", "scoreAway"
] if c in df.columns]

if not idxs:
    print("No exact match found. Nearby unique clocks in period 1:")
    print(df.loc[df["period"] == target_period, "clock"].dropna().drop_duplicates().head(20).tolist())
else:
    for i, idx in enumerate(idxs, start=1):
        start = max(0, idx - 3)
        end = min(len(df) - 1, idx + 3)
        print(f"\n--- Match {i} at index={idx}, showing rows {start}..{end} ---")
        display(df.loc[start:end, cols])

matches for (period=1, clock=PT04M50.00S): 5

--- Match 1 at index=65, showing rows 62..68 ---


,actionNumber,actionId,period,clock,teamId,teamTricode,personId,playerName,actionType,subType,description,scoreHome,scoreAway
62,83,63,1,PT05M32.00S,1610612738,BOS,202696,Vučević,Rebound,Unknown,Vucevic REBOUND (Off:0 Def:1),,
63,84,64,1,PT05M21.00S,1610612738,BOS,1627759,Brown,Made Shot,Driving Layup Shot,Brown 1' Driving Layup (4 PTS) (White 2 AST),22,9
64,86,65,1,PT05M04.00S,1610612755,PHI,202331,George,Made Shot,Fadeaway Jump Shot,George 14' Fadeaway Jumper (5 PTS) (Drummond 1...,22,11
65,88,66,1,PT04M50.00S,1610612738,BOS,202696,Vučević,Foul,Offensive,Vucevic OFF.Foul (P1) (P.Fraher),,
66,90,67,1,PT04M50.00S,1610612738,BOS,202696,Vučević,Turnover,Offensive Foul Turnover,Vucevic Offensive Foul Turnover (P1.T3),,
67,91,68,1,PT04M50.00S,1610612738,BOS,1630573,Hauser,Substitution,,SUB: Pritchard FOR Hauser,,
68,92,69,1,PT04M50.00S,1610612755,PHI,1626162,Oubre Jr.,Substitution,,SUB: Grimes FOR Oubre Jr.,,



--- Match 2 at index=66, showing rows 63..69 ---


,actionNumber,actionId,period,clock,teamId,teamTricode,personId,playerName,actionType,subType,description,scoreHome,scoreAway
63,84,64,1,PT05M21.00S,1610612738,BOS,1627759,Brown,Made Shot,Driving Layup Shot,Brown 1' Driving Layup (4 PTS) (White 2 AST),22,9
64,86,65,1,PT05M04.00S,1610612755,PHI,202331,George,Made Shot,Fadeaway Jump Shot,George 14' Fadeaway Jumper (5 PTS) (Drummond 1...,22,11
65,88,66,1,PT04M50.00S,1610612738,BOS,202696,Vučević,Foul,Offensive,Vucevic OFF.Foul (P1) (P.Fraher),,
66,90,67,1,PT04M50.00S,1610612738,BOS,202696,Vučević,Turnover,Offensive Foul Turnover,Vucevic Offensive Foul Turnover (P1.T3),,
67,91,68,1,PT04M50.00S,1610612738,BOS,1630573,Hauser,Substitution,,SUB: Pritchard FOR Hauser,,
68,92,69,1,PT04M50.00S,1610612755,PHI,1626162,Oubre Jr.,Substitution,,SUB: Grimes FOR Oubre Jr.,,
69,93,70,1,PT04M50.00S,1610612755,PHI,1642845,Edgecombe,Substitution,,SUB: Barlow FOR Edgecombe,,



--- Match 3 at index=67, showing rows 64..70 ---


,actionNumber,actionId,period,clock,teamId,teamTricode,personId,playerName,actionType,subType,description,scoreHome,scoreAway
64,86,65,1,PT05M04.00S,1610612755,PHI,202331,George,Made Shot,Fadeaway Jump Shot,George 14' Fadeaway Jumper (5 PTS) (Drummond 1...,22,11
65,88,66,1,PT04M50.00S,1610612738,BOS,202696,Vučević,Foul,Offensive,Vucevic OFF.Foul (P1) (P.Fraher),,
66,90,67,1,PT04M50.00S,1610612738,BOS,202696,Vučević,Turnover,Offensive Foul Turnover,Vucevic Offensive Foul Turnover (P1.T3),,
67,91,68,1,PT04M50.00S,1610612738,BOS,1630573,Hauser,Substitution,,SUB: Pritchard FOR Hauser,,
68,92,69,1,PT04M50.00S,1610612755,PHI,1626162,Oubre Jr.,Substitution,,SUB: Grimes FOR Oubre Jr.,,
69,93,70,1,PT04M50.00S,1610612755,PHI,1642845,Edgecombe,Substitution,,SUB: Barlow FOR Edgecombe,,
70,97,71,1,PT04M42.00S,1610612738,BOS,1627759,Brown,Foul,Personal,Brown P.FOUL (P1.T3) (T.Maddox),,



--- Match 4 at index=68, showing rows 65..71 ---


,actionNumber,actionId,period,clock,teamId,teamTricode,personId,playerName,actionType,subType,description,scoreHome,scoreAway
65,88,66,1,PT04M50.00S,1610612738,BOS,202696,Vučević,Foul,Offensive,Vucevic OFF.Foul (P1) (P.Fraher),,
66,90,67,1,PT04M50.00S,1610612738,BOS,202696,Vučević,Turnover,Offensive Foul Turnover,Vucevic Offensive Foul Turnover (P1.T3),,
67,91,68,1,PT04M50.00S,1610612738,BOS,1630573,Hauser,Substitution,,SUB: Pritchard FOR Hauser,,
68,92,69,1,PT04M50.00S,1610612755,PHI,1626162,Oubre Jr.,Substitution,,SUB: Grimes FOR Oubre Jr.,,
69,93,70,1,PT04M50.00S,1610612755,PHI,1642845,Edgecombe,Substitution,,SUB: Barlow FOR Edgecombe,,
70,97,71,1,PT04M42.00S,1610612738,BOS,1627759,Brown,Foul,Personal,Brown P.FOUL (P1.T3) (T.Maddox),,
71,99,72,1,PT04M39.00S,1610612755,PHI,203083,Drummond,Foul,Offensive,Drummond OFF.Foul (P2) (S.Foster),,



--- Match 5 at index=69, showing rows 66..72 ---


,actionNumber,actionId,period,clock,teamId,teamTricode,personId,playerName,actionType,subType,description,scoreHome,scoreAway
66,90,67,1,PT04M50.00S,1610612738,BOS,202696,Vučević,Turnover,Offensive Foul Turnover,Vucevic Offensive Foul Turnover (P1.T3),,
67,91,68,1,PT04M50.00S,1610612738,BOS,1630573,Hauser,Substitution,,SUB: Pritchard FOR Hauser,,
68,92,69,1,PT04M50.00S,1610612755,PHI,1626162,Oubre Jr.,Substitution,,SUB: Grimes FOR Oubre Jr.,,
69,93,70,1,PT04M50.00S,1610612755,PHI,1642845,Edgecombe,Substitution,,SUB: Barlow FOR Edgecombe,,
70,97,71,1,PT04M42.00S,1610612738,BOS,1627759,Brown,Foul,Personal,Brown P.FOUL (P1.T3) (T.Maddox),,
71,99,72,1,PT04M39.00S,1610612755,PHI,203083,Drummond,Foul,Offensive,Drummond OFF.Foul (P2) (S.Foster),,
72,101,73,1,PT04M39.00S,1610612755,PHI,203083,Drummond,Turnover,Offensive Foul Turnover,Drummond Offensive Foul Turnover (P1.T5),,


In [21]:
sub_mask = pbp_df["actionType"].astype(str).str.lower().eq("substitution")
subs_df = pbp_df.loc[sub_mask].copy()

print("Total substitution rows:", len(subs_df))

sub_groups = (
    subs_df.groupby(["period", "clock"])
    .size()
    .reset_index(name="num_sub_rows")
    .sort_values(["period", "clock"], ascending=[True, False])
)

display(sub_groups.head(20))

print("\nGroups with more than 1 substitution row:")
display(sub_groups[sub_groups["num_sub_rows"] > 1].head(20))

Total substitution rows: 45


,period,clock,num_sub_rows
6,1,PT10M23.00S,1
5,1,PT08M39.00S,1
4,1,PT04M50.00S,3
3,1,PT04M39.00S,1
2,1,PT03M49.00S,1
1,1,PT02M39.00S,1
0,1,PT00M27.30S,2
16,2,PT08M43.00S,3
15,2,PT07M16.00S,2
14,2,PT06M50.00S,1



Groups with more than 1 substitution row:


,period,clock,num_sub_rows
4,1,PT04M50.00S,3
0,1,PT00M27.30S,2
16,2,PT08M43.00S,3
15,2,PT07M16.00S,2
20,3,PT04M33.00S,2
19,3,PT03M36.00S,3
17,3,PT00M43.30S,2
30,4,PT09M16.00S,2
27,4,PT07M02.00S,2
24,4,PT04M10.00S,3


In [22]:
import re
import pandas as pd

sub_df = pbp_df[pbp_df["actionType"].astype(str).str.lower() == "substitution"].copy()

def parse_sub_description(desc):
    """
    Parse strings like:
    'SUB: Pritchard FOR Hauser'
    Returns (player_in_name, player_out_name_from_text)
    """
    if pd.isna(desc):
        return None, None
    
    desc = str(desc).strip()
    m = re.search(r"SUB:\s*(.*?)\s+FOR\s+(.*)", desc, flags=re.IGNORECASE)
    if m:
        player_in = m.group(1).strip()
        player_out = m.group(2).strip()
        return player_in, player_out
    return None, None

sub_df[["player_in_name", "player_out_name_text"]] = sub_df["description"].apply(
    lambda x: pd.Series(parse_sub_description(x))
)

sub_df["player_out_id"] = sub_df["personId"]
sub_df["player_out_name_api"] = sub_df["playerName"]

display(sub_df[[
    c for c in [
        "actionNumber", "period", "clock", "teamId", "teamTricode",
        "player_out_id", "player_out_name_api",
        "player_in_name", "player_out_name_text",
        "description"
    ] if c in sub_df.columns
]].head(20))

,actionNumber,period,clock,teamId,teamTricode,player_out_id,player_out_name_api,player_in_name,player_out_name_text,description
16,24,1,PT10M23.00S,1610612755,PHI,1641737,Bona,Drummond,Bona,SUB: Drummond FOR Bona
34,49,1,PT08M39.00S,1610612738,BOS,1629674,Queta,Vucevic,Queta,SUB: Vucevic FOR Queta
67,91,1,PT04M50.00S,1610612738,BOS,1630573,Hauser,Pritchard,Hauser,SUB: Pritchard FOR Hauser
68,92,1,PT04M50.00S,1610612755,PHI,1626162,Oubre Jr.,Grimes,Oubre Jr.,SUB: Grimes FOR Oubre Jr.
69,93,1,PT04M50.00S,1610612755,PHI,1642845,Edgecombe,Barlow,Edgecombe,SUB: Barlow FOR Edgecombe
74,106,1,PT04M39.00S,1610612755,PHI,203083,Drummond,Oubre Jr.,Drummond,SUB: Oubre Jr. FOR Drummond
85,120,1,PT03M49.00S,1610612738,BOS,1627759,Brown,Walsh,Brown,SUB: Walsh FOR Brown
96,133,1,PT02M39.00S,1610612755,PHI,202331,George,Edwards,George,SUB: Edwards FOR George
114,157,1,PT00M27.30S,1610612738,BOS,202696,Vučević,Brown,Vucevic,SUB: Brown FOR Vucevic
115,158,1,PT00M27.30S,1610612755,PHI,1631230,Barlow,George,Barlow,SUB: George FOR Barlow


In [23]:
sub_df["clock_key"] = sub_df["clock"].astype(str)

sub_groups = (
    sub_df.groupby(["period", "clock_key"], sort=False)
    .apply(lambda g: g[[
        "actionNumber",
        "teamId",
        "teamTricode",
        "player_out_id",
        "player_out_name_api",
        "player_in_name",
        "description"
    ]].to_dict("records"))
    .to_dict()
)

# inspect one example
sub_groups[(1, "PT04M50.00S")]

[{'actionNumber': 91,
  'teamId': 1610612738,
  'teamTricode': 'BOS',
  'player_out_id': 1630573,
  'player_out_name_api': 'Hauser',
  'player_in_name': 'Pritchard',
  'description': 'SUB: Pritchard FOR Hauser'},
 {'actionNumber': 92,
  'teamId': 1610612755,
  'teamTricode': 'PHI',
  'player_out_id': 1626162,
  'player_out_name_api': 'Oubre Jr.',
  'player_in_name': 'Grimes',
  'description': 'SUB: Grimes FOR Oubre Jr.'},
 {'actionNumber': 93,
  'teamId': 1610612755,
  'teamTricode': 'PHI',
  'player_out_id': 1642845,
  'player_out_name_api': 'Edgecombe',
  'player_in_name': 'Barlow',
  'description': 'SUB: Barlow FOR Edgecombe'}]

In [24]:
import unicodedata

def normalize_name(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    return x

sub_df["player_out_name_api_norm"] = sub_df["player_out_name_api"].apply(normalize_name)
sub_df["player_out_name_text_norm"] = sub_df["player_out_name_text"].apply(normalize_name)

sub_df["out_names_match_norm"] = (
    sub_df["player_out_name_api_norm"] == sub_df["player_out_name_text_norm"]
)

print(sub_df["out_names_match_norm"].value_counts(dropna=False))

out_names_match_norm
True    45
Name: count, dtype: int64


In [25]:
# Cell 1: Inspect and group substitution rows by (period, clock)
import re
import pandas as pd

# 1) Keep a stable event order
work_df = pbp_df.sort_values("actionNumber", kind="mergesort").reset_index(drop=True).copy()

# 2) Substitution rows
subs_df = work_df[work_df["actionType"].eq("Substitution")].copy()

# 3) Parse "SUB: X FOR Y"
sub_pattern = re.compile(r"^SUB:\s*(?P<player_in>.+?)\s+FOR\s+(?P<player_out>.+?)\s*$", re.IGNORECASE)

parsed = subs_df["description"].astype(str).str.extract(sub_pattern)
subs_df["player_in_name"] = parsed["player_in"].str.strip()
subs_df["player_out_name_from_desc"] = parsed["player_out"].str.strip()

# Optional QA: outgoing in description vs outgoing in personId/playerName columns
subs_df["player_out_name_from_row"] = subs_df["playerName"].astype(str).str.strip()
subs_df["parsed_ok"] = subs_df["player_in_name"].notna() & subs_df["player_out_name_from_desc"].notna()

print(f"Total substitution rows: {len(subs_df):,}")
print(f"Parse failures: {(~subs_df['parsed_ok']).sum():,}")

# 4) Show grouped substitutions by timestamp
group_cols = ["period", "clock"]
subs_group_summary = (
    subs_df.groupby(group_cols, as_index=False)
    .agg(
        n_subs=("actionNumber", "size"),
        action_numbers=("actionNumber", lambda s: list(s)),
        team_ids=("teamId", lambda s: list(s)),
        players_in=("player_in_name", lambda s: list(s)),
        players_out=("player_out_name_from_desc", lambda s: list(s)),
    )
    .sort_values(["period", "clock"], ascending=[True, False])  # clock is time remaining
)

display(subs_group_summary.head(30))

# 5) Only show problematic timestamps with multiple substitutions
multi_sub_groups = subs_group_summary[subs_group_summary["n_subs"] > 1].copy()
print(f"Timestamps with multiple substitutions: {len(multi_sub_groups):,}")
display(multi_sub_groups.head(30))

Total substitution rows: 45
Parse failures: 0


,period,clock,n_subs,action_numbers,team_ids,players_in,players_out
6,1,PT10M23.00S,1,[24],[1610612755],[Drummond],[Bona]
5,1,PT08M39.00S,1,[49],[1610612738],[Vucevic],[Queta]
4,1,PT04M50.00S,3,"[91, 92, 93]","[1610612738, 1610612755, 1610612755]","[Pritchard, Grimes, Barlow]","[Hauser, Oubre Jr., Edgecombe]"
3,1,PT04M39.00S,1,[106],[1610612755],[Oubre Jr.],[Drummond]
2,1,PT03M49.00S,1,[120],[1610612738],[Walsh],[Brown]
1,1,PT02M39.00S,1,[133],[1610612755],[Edwards],[George]
0,1,PT00M27.30S,2,"[157, 158]","[1610612738, 1610612755]","[Brown, George]","[Vucevic, Barlow]"
16,2,PT08M43.00S,3,"[221, 222, 223]","[1610612738, 1610612738, 1610612755]","[White, Queta, Maxey]","[Garza, Scheierman, Grimes]"
15,2,PT07M16.00S,2,"[241, 242]","[1610612738, 1610612755]","[Tatum, Bona]","[Hauser, Drummond]"
14,2,PT06M50.00S,1,[250],[1610612755],[Oubre Jr.],[George]


Timestamps with multiple substitutions: 10


,period,clock,n_subs,action_numbers,team_ids,players_in,players_out
4,1,PT04M50.00S,3,"[91, 92, 93]","[1610612738, 1610612755, 1610612755]","[Pritchard, Grimes, Barlow]","[Hauser, Oubre Jr., Edgecombe]"
0,1,PT00M27.30S,2,"[157, 158]","[1610612738, 1610612755]","[Brown, George]","[Vucevic, Barlow]"
16,2,PT08M43.00S,3,"[221, 222, 223]","[1610612738, 1610612738, 1610612755]","[White, Queta, Maxey]","[Garza, Scheierman, Grimes]"
15,2,PT07M16.00S,2,"[241, 242]","[1610612738, 1610612755]","[Tatum, Bona]","[Hauser, Drummond]"
20,3,PT04M33.00S,2,"[451, 452]","[1610612755, 1610612755]","[Grimes, Bona]","[Drummond, Oubre Jr.]"
19,3,PT03M36.00S,3,"[467, 468, 471]","[1610612738, 1610612738, 1610612755]","[Walsh, Garza, Barlow]","[Tatum, Vucevic, George]"
17,3,PT00M43.30S,2,"[506, 507]","[1610612738, 1610612755]","[Hauser, Edwards]","[Garza, Bona]"
30,4,PT09M16.00S,2,"[558, 559]","[1610612755, 1610612755]","[Drummond, Oubre Jr.]","[Edwards, Barlow]"
27,4,PT07M02.00S,2,"[587, 588]","[1610612755, 1610612755]","[Watford, Bona]","[Drummond, Edgecombe]"
24,4,PT04M10.00S,3,"[636, 637, 638]","[1610612738, 1610612755, 1610612755]","[Harper Jr., Walker, Barlow]","[Hauser, Edwards, Bona]"


In [26]:
# Cell 2: Reusable helper - get grouped substitution events
import re
import pandas as pd

SUB_PATTERN = re.compile(r"^SUB:\s*(?P<player_in>.+?)\s+FOR\s+(?P<player_out>.+?)\s*$", re.IGNORECASE)

def get_substitution_groups(pbp_df: pd.DataFrame) -> pd.DataFrame:
    """
    Return substitution events grouped by (period, clock), preserving feed order.

    Output columns:
      - period, clock
      - n_subs
      - substitutions: list[dict] with actionNumber/actionId/teamId/personId/playerName/in/out/description
    """
    df = pbp_df.sort_values("actionNumber", kind="mergesort").reset_index(drop=True).copy()
    subs = df[df["actionType"].eq("Substitution")].copy()

    parsed = subs["description"].astype(str).str.extract(SUB_PATTERN)
    subs["player_in_name"] = parsed["player_in"].str.strip()
    subs["player_out_name_from_desc"] = parsed["player_out"].str.strip()

    # Build event dict per substitution row (easy to inspect/debug in notebook)
    event_cols = [
        "actionNumber", "actionId", "period", "clock", "teamId", "teamTricode",
        "personId", "playerName", "player_in_name", "player_out_name_from_desc", "description"
    ]
    for c in event_cols:
        if c not in subs.columns:
            subs[c] = pd.NA

    subs["sub_event"] = subs[event_cols].to_dict(orient="records")

    grouped = (
        subs.groupby(["period", "clock"], as_index=False)
        .agg(
            n_subs=("sub_event", "size"),
            substitutions=("sub_event", list),
        )
        .sort_values(["period", "clock"], ascending=[True, False])
        .reset_index(drop=True)
    )
    return grouped

# Example usage
sub_groups_df = get_substitution_groups(pbp_df)
display(sub_groups_df.head(20))
display(sub_groups_df[sub_groups_df["n_subs"] > 1].head(20))

,period,clock,n_subs,substitutions
0,1,PT10M23.00S,1,"[{'actionNumber': 24, 'actionId': 17, 'period'..."
1,1,PT08M39.00S,1,"[{'actionNumber': 49, 'actionId': 37, 'period'..."
2,1,PT04M50.00S,3,"[{'actionNumber': 91, 'actionId': 68, 'period'..."
3,1,PT04M39.00S,1,"[{'actionNumber': 106, 'actionId': 75, 'period..."
4,1,PT03M49.00S,1,"[{'actionNumber': 120, 'actionId': 86, 'period..."
5,1,PT02M39.00S,1,"[{'actionNumber': 133, 'actionId': 97, 'period..."
6,1,PT00M27.30S,2,"[{'actionNumber': 157, 'actionId': 115, 'perio..."
7,2,PT08M43.00S,3,"[{'actionNumber': 221, 'actionId': 157, 'perio..."
8,2,PT07M16.00S,2,"[{'actionNumber': 241, 'actionId': 171, 'perio..."
9,2,PT06M50.00S,1,"[{'actionNumber': 250, 'actionId': 176, 'perio..."


,period,clock,n_subs,substitutions
2,1,PT04M50.00S,3,"[{'actionNumber': 91, 'actionId': 68, 'period'..."
6,1,PT00M27.30S,2,"[{'actionNumber': 157, 'actionId': 115, 'perio..."
7,2,PT08M43.00S,3,"[{'actionNumber': 221, 'actionId': 157, 'perio..."
8,2,PT07M16.00S,2,"[{'actionNumber': 241, 'actionId': 171, 'perio..."
19,3,PT04M33.00S,2,"[{'actionNumber': 451, 'actionId': 327, 'perio..."
20,3,PT03M36.00S,3,"[{'actionNumber': 467, 'actionId': 337, 'perio..."
22,3,PT00M43.30S,2,"[{'actionNumber': 506, 'actionId': 363, 'perio..."
23,4,PT09M16.00S,2,"[{'actionNumber': 558, 'actionId': 398, 'perio..."
26,4,PT07M02.00S,2,"[{'actionNumber': 587, 'actionId': 418, 'perio..."
29,4,PT04M10.00S,3,"[{'actionNumber': 636, 'actionId': 453, 'perio..."


In [27]:
import re
import pandas as pd

def clock_to_seconds_remaining(clock_str):
    if pd.isna(clock_str):
        return None
    m = re.match(r"PT(?:(\d+)M)?([0-9.]+)S", str(clock_str))
    if not m:
        return None
    minutes = int(m.group(1)) if m.group(1) else 0
    seconds = float(m.group(2))
    return minutes * 60 + seconds

pbp_df["clock_seconds_remaining"] = pbp_df["clock"].apply(clock_to_seconds_remaining)

# all event timestamp groups in game order
event_groups_df = (
    pbp_df.groupby(["period", "clock", "clock_seconds_remaining"], sort=False)
    .agg(
        n_events=("actionNumber", "count"),
        action_numbers=("actionNumber", list),
        action_types=("actionType", list),
    )
    .reset_index()
    .sort_values(
        ["period", "clock_seconds_remaining"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

# mark which timestamp groups contain substitutions
sub_keys = set(zip(sub_groups_df["period"], sub_groups_df["clock"]))

event_groups_df["has_substitution"] = event_groups_df.apply(
    lambda row: (row["period"], row["clock"]) in sub_keys,
    axis=1
)

display(event_groups_df.head(25))

,period,clock,clock_seconds_remaining,n_events,action_numbers,action_types,has_substitution
0,1,PT12M00.00S,720.0,2,"[2, 4]","[period, Jump Ball]",False
1,1,PT11M48.00S,708.0,1,[7],[Missed Shot],False
2,1,PT11M47.00S,707.0,1,[8],[Rebound],False
3,1,PT11M38.00S,698.0,4,"[9, 11, 12, 13]","[Foul, Free Throw, Free Throw, Rebound]",False
4,1,PT11M24.00S,684.0,1,[14],[Missed Shot],False
5,1,PT11M23.00S,683.0,1,[15],[Rebound],False
6,1,PT11M15.00S,675.0,1,[16],[Made Shot],False
7,1,PT10M57.00S,657.0,1,[18],[Made Shot],False
8,1,PT10M29.00S,629.0,1,[19],[Missed Shot],False
9,1,PT10M28.00S,628.0,1,[20],[Rebound],False


In [31]:
# Build a quick lookup from (period, clock) -> substitution batch
sub_group_lookup = {
    (row["period"], row["clock"]): row["substitutions"]
    for _, row in sub_groups_df.iterrows()
}

# Placeholder current lineup state
current_home_lineup = None
current_away_lineup = None

# We'll only track stint boundaries for now, not final perfect lineups yet
stint_debug_rows = []

current_stint_start_period = None
current_stint_start_clock = None

for i, row in event_groups_df.iterrows():
    period = row["period"]
    clock = row["clock"]
    clock_sec = row["clock_seconds_remaining"]
    has_sub = row["has_substitution"]

    # initialize first stint start
    if current_stint_start_period is None:
        current_stint_start_period = period
        current_stint_start_clock = clock

    # if this timestamp group has substitutions, close old stint once
    if has_sub:
        # close previous stint only if time has actually moved
        if not (current_stint_start_period == period and current_stint_start_clock == clock):
            stint_debug_rows.append({
                "start_period": current_stint_start_period,
                "start_clock": current_stint_start_clock,
                "end_period": period,
                "end_clock": clock,
                "ended_by_substitution": True,
                "num_subs_applied": len(sub_group_lookup[(period, clock)])
            })

        # apply all substitutions in this timestamp group together
        sub_batch = sub_group_lookup[(period, clock)]

        # DEBUG: inspect what would be applied
        print(f"\nApplying substitution batch at P{period} {clock}")
        for sub in sub_batch:
            print(
                f"  Team {sub.get('teamTricode', sub.get('teamId'))}: "
                f"{sub.get('player_out_name_api')} OUT -> {sub.get('player_in_name')} IN"
            )

        # start exactly one new stint at this same timestamp
        current_stint_start_period = period
        current_stint_start_clock = clock

# show debug stint boundaries
stint_debug_df = pd.DataFrame(stint_debug_rows)
display(stint_debug_df.head(20))
print("Number of debug stints:", len(stint_debug_df))


Applying substitution batch at P1 PT10M23.00S
  Team PHI: Bona OUT -> Drummond IN

Applying substitution batch at P1 PT08M39.00S
  Team BOS: Queta OUT -> Vucevic IN

Applying substitution batch at P1 PT04M50.00S
  Team BOS: Hauser OUT -> Pritchard IN
  Team PHI: Oubre Jr. OUT -> Grimes IN
  Team PHI: Edgecombe OUT -> Barlow IN

Applying substitution batch at P1 PT04M39.00S
  Team PHI: Drummond OUT -> Oubre Jr. IN

Applying substitution batch at P1 PT03M49.00S
  Team BOS: Brown OUT -> Walsh IN

Applying substitution batch at P1 PT02M39.00S
  Team PHI: George OUT -> Edwards IN

Applying substitution batch at P1 PT00M27.30S
  Team BOS: Vučević OUT -> Brown IN
  Team PHI: Barlow OUT -> George IN

Applying substitution batch at P2 PT08M43.00S
  Team BOS: Garza OUT -> White IN
  Team BOS: Scheierman OUT -> Queta IN
  Team PHI: Grimes OUT -> Maxey IN

Applying substitution batch at P2 PT07M16.00S
  Team BOS: Hauser OUT -> Tatum IN
  Team PHI: Drummond OUT -> Bona IN

Applying substitution ba

,start_period,start_clock,end_period,end_clock,ended_by_substitution,num_subs_applied
0,1,PT12M00.00S,1,PT10M23.00S,True,1
1,1,PT10M23.00S,1,PT08M39.00S,True,1
2,1,PT08M39.00S,1,PT04M50.00S,True,3
3,1,PT04M50.00S,1,PT04M39.00S,True,1
4,1,PT04M39.00S,1,PT03M49.00S,True,1
5,1,PT03M49.00S,1,PT02M39.00S,True,1
6,1,PT02M39.00S,1,PT00M27.30S,True,2
7,1,PT00M27.30S,2,PT08M43.00S,True,3
8,2,PT08M43.00S,2,PT07M16.00S,True,2
9,2,PT07M16.00S,2,PT06M50.00S,True,1


Number of debug stints: 31


In [ ]:
sub_event_cols = [
    "actionNumber",
    "actionId",
    "period",
    "clock",
    "teamId",
    "teamTricode",
    "player_out_id",
    "player_out_name_api",
    "player_in_name",
    "player_out_name_text",
    "description"
]

sub_events_clean = (
    sub_df[sub_event_cols]
    .sort_values(["period", "clock", "actionNumber"], ascending=[True, False, True])
    .reset_index(drop=True)
)

sub_groups_df = (
    sub_events_clean.groupby(["period", "clock"], sort=False)
    .apply(lambda g: pd.Series({
        "n_subs": len(g),
        "substitutions": g.to_dict("records")
    }))
    .reset_index()
)

display(sub_groups_df.head(10))

,period,clock,n_subs,substitutions
0,1,PT10M23.00S,1,"[{'actionNumber': 24, 'actionId': 17, 'teamId'..."
1,1,PT08M39.00S,1,"[{'actionNumber': 49, 'actionId': 37, 'teamId'..."
2,1,PT04M50.00S,3,"[{'actionNumber': 91, 'actionId': 68, 'teamId'..."
3,1,PT04M39.00S,1,"[{'actionNumber': 106, 'actionId': 75, 'teamId..."
4,1,PT03M49.00S,1,"[{'actionNumber': 120, 'actionId': 86, 'teamId..."
5,1,PT02M39.00S,1,"[{'actionNumber': 133, 'actionId': 97, 'teamId..."
6,1,PT00M27.30S,2,"[{'actionNumber': 157, 'actionId': 115, 'teamI..."
7,2,PT08M43.00S,3,"[{'actionNumber': 221, 'actionId': 157, 'teamI..."
8,2,PT07M16.00S,2,"[{'actionNumber': 241, 'actionId': 171, 'teamI..."
9,2,PT06M50.00S,1,"[{'actionNumber': 250, 'actionId': 176, 'teamI..."


In [ ]:
sub_group_lookup = {
    (row["period"], row["clock"]): row["substitutions"]
    for _, row in sub_groups_df.iterrows()
}

In [32]:
# Cell 1: Manually seed one game's starting state (NAMES, not IDs)

# Example IDs from your test game (change if needed)
home_team_id = 1610612738  # BOS
away_team_id = 1610612755  # PHI

# Manually provide starting 5 by player LAST/FULL names matching your parsed batches
current_home_lineup = {"Hauser", "Tatum", "Queta", "Brown", "White"}
current_away_lineup = {"George", "Oubre Jr.", "Bona", "Edgecombe", "Maxey"}

print("Initial Home Lineup:", sorted(current_home_lineup))
print("Initial Away Lineup:", sorted(current_away_lineup))
print("Home count:", len(current_home_lineup), "| Away count:", len(current_away_lineup))

Initial Home Lineup: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
Initial Away Lineup: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
Home count: 5 | Away count: 5


In [33]:
# Cell 2: Helper to apply one substitution batch at one (period, clock)

def apply_substitution_batch(
    home_lineup: set[str],
    away_lineup: set[str],
    sub_batch: list[dict],
    home_team_id: int,
    away_team_id: int,
) -> tuple[set[str], set[str]]:
    """
    Apply all substitutions in a single timestamp batch.
    Each sub dict should contain at least:
      - teamId
      - player_in_name
      - player_out_name
      - description (optional, for debugging)
    """
    # Work on copies so notebook state is predictable
    home = set(home_lineup)
    away = set(away_lineup)

    for sub in sub_batch:
        team_id = int(sub["teamId"])
        player_in = str(sub["player_in_name"]).strip()
        player_out = str(sub["player_out_name"]).strip()

        if team_id == home_team_id:
            target = home
            side = "HOME"
        elif team_id == away_team_id:
            target = away
            side = "AWAY"
        else:
            print(f"[WARN] Unknown teamId={team_id}; skipping sub: {sub.get('description', sub)}")
            continue

        # Outgoing must be on floor for a clean update
        if player_out not in target:
            print(f"[WARN] {side} outgoing not found: {player_out} | current={sorted(target)}")
            # Keep going; still try to add incoming for debugging visibility

        else:
            target.remove(player_out)

        # Incoming ideally not already on floor
        if player_in in target:
            print(f"[WARN] {side} incoming already on floor: {player_in}")
        else:
            target.add(player_in)

    return home, away

In [34]:
# Cell 3: Test one known batch manually (example: period 1, PT04M50.00S)

# Example structure expected by helper (replace names with your actual parsed batch)
test_batch = [
    # {"teamId": 1610612738, "player_in_name": "Pritchard", "player_out_name": "Hauser", "description": "SUB: Pritchard FOR Hauser"},
    # {"teamId": 1610612755, "player_in_name": "Grimes", "player_out_name": "Oubre Jr.", "description": "SUB: Grimes FOR Oubre Jr."},
]

print("Before batch:")
print("  HOME:", sorted(current_home_lineup))
print("  AWAY:", sorted(current_away_lineup))

new_home, new_away = apply_substitution_batch(
    current_home_lineup,
    current_away_lineup,
    test_batch,
    home_team_id,
    away_team_id,
)

print("\nAfter batch:")
print("  HOME:", sorted(new_home), "| count:", len(new_home))
print("  AWAY:", sorted(new_away), "| count:", len(new_away))

Before batch:
  HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
  AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']

After batch:
  HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White'] | count: 5
  AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.'] | count: 5


In [37]:
# Cell 4: Loop through all Q1 substitution batches in order and validate lineup state

# Expected input: a grouped substitutions DataFrame from your earlier parsing step.
# It should have:
#   period, clock, substitutions
# where substitutions is list[dict] with teamId/player_in_name/player_out_name

# Example:
# sub_groups_df = get_substitution_groups(pbp_df)

q1_groups = (
    sub_groups_df[sub_groups_df["period"] == 1]
    .copy()
    .sort_values("clock", ascending=False)  # clock is time remaining
    .reset_index(drop=True)
)

home_state = set(current_home_lineup)
away_state = set(current_away_lineup)

print("=== Q1 substitution batch replay ===")
print("Start HOME:", sorted(home_state))
print("Start AWAY:", sorted(away_state))
print()

for i, row in q1_groups.iterrows():
    clock = row["clock"]
    batch = row["substitutions"]

    print(f"[Batch {i+1}] period=1 clock={clock} n_subs={len(batch)}")

    home_state, away_state = apply_substitution_batch(
        home_state,
        away_state,
        batch,
        home_team_id,
        away_team_id,
    )

    # Hard validation checks
    if len(home_state) != 5:
        print(f"[ERROR] HOME lineup size != 5 at {clock}: {len(home_state)}")
    if len(away_state) != 5:
        print(f"[ERROR] AWAY lineup size != 5 at {clock}: {len(away_state)}")

    print("  HOME:", sorted(home_state))
    print("  AWAY:", sorted(away_state))
    print("-" * 80)

print("Final HOME:", sorted(home_state), "| count:", len(home_state))
print("Final AWAY:", sorted(away_state), "| count:", len(away_state))

=== Q1 substitution batch replay ===
Start HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
Start AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']

[Batch 1] period=1 clock=PT10M23.00S n_subs=1
  HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
  AWAY: ['Drummond', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
--------------------------------------------------------------------------------
[Batch 2] period=1 clock=PT08M39.00S n_subs=1
  HOME: ['Brown', 'Hauser', 'Tatum', 'Vucevic', 'White']
  AWAY: ['Drummond', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
--------------------------------------------------------------------------------
[Batch 3] period=1 clock=PT04M50.00S n_subs=3
  HOME: ['Brown', 'Pritchard', 'Tatum', 'Vucevic', 'White']
  AWAY: ['Barlow', 'Drummond', 'George', 'Grimes', 'Maxey']
--------------------------------------------------------------------------------
[Batch 4] period=1 clock=PT04M39.00S n_subs=1
  HOME: ['Brown', 'Pritchard', 'Tatum', 'Vucev

In [36]:
import unicodedata

def normalize_name(name):
    if name is None:
        return None
    name = str(name).strip()
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("utf-8")
    return name.lower()

def apply_substitution_batch(home_lineup, away_lineup, sub_batch, home_team_id, away_team_id):
    home = set(home_lineup)
    away = set(away_lineup)

    # normalized lookup maps
    home_lookup = {normalize_name(p): p for p in home}
    away_lookup = {normalize_name(p): p for p in away}

    for sub in sub_batch:
        team_id = int(sub["teamId"])
        player_in = str(sub.get("player_in_name", "")).strip()

        # use the correct outgoing field
        player_out = str(
            sub.get("player_out_name_api") or sub.get("player_out_name_text") or ""
        ).strip()

        player_out_norm = normalize_name(player_out)

        if team_id == home_team_id:
            if player_out_norm in home_lookup:
                original_name = home_lookup[player_out_norm]
                home.remove(original_name)
            else:
                print(f"[WARN] HOME outgoing player not found: {player_out}")

            home.add(player_in)

        elif team_id == away_team_id:
            if player_out_norm in away_lookup:
                original_name = away_lookup[player_out_norm]
                away.remove(original_name)
            else:
                print(f"[WARN] AWAY outgoing player not found: {player_out}")

            away.add(player_in)

        else:
            print(f"[WARN] Unknown team_id: {team_id}")

    return home, away

In [38]:
# Cell 1: helpers (score snapshot + append stint row)

import pandas as pd

def get_score_snapshot_for_group(pbp_df, period, clock):
    """
    Return (score_home, score_away) for a timestamp group.
    Uses all rows at (period, clock), forward-filling score within the slice.
    """
    rows = pbp_df[(pbp_df["period"] == period) & (pbp_df["clock"] == clock)].copy()
    if rows.empty:
        return None, None

    # Keep feed order
    if "actionNumber" in rows.columns:
        rows = rows.sort_values("actionNumber", kind="mergesort")

    rows["scoreHome"] = pd.to_numeric(rows["scoreHome"], errors="coerce").ffill()
    rows["scoreAway"] = pd.to_numeric(rows["scoreAway"], errors="coerce").ffill()

    # Last row at that timestamp is the best "as-of timestamp" snapshot
    sh = rows["scoreHome"].iloc[-1]
    sa = rows["scoreAway"].iloc[-1]

    return (None if pd.isna(sh) else int(sh),
            None if pd.isna(sa) else int(sa))


def append_stint_row(
    out_rows,
    period,
    start_clock,
    end_clock,
    start_sec,
    end_sec,
    home_lineup,
    away_lineup,
    score_home_start,
    score_away_start,
    score_home_end,
    score_away_end,
    ended_by_substitution,
):
    """
    Append one stint row. Duration is computed as start_sec - end_sec (clock counts down).
    """
    duration = max(0.0, float(start_sec) - float(end_sec))

    out_rows.append({
        "period": int(period),
        "start_clock": start_clock,
        "end_clock": end_clock,
        "duration_seconds": duration,
        "home_lineup": sorted(home_lineup),   # keep as list for readability/debug
        "away_lineup": sorted(away_lineup),
        "score_home_start": score_home_start,
        "score_away_start": score_away_start,
        "score_home_end": score_home_end,
        "score_away_end": score_away_end,
        "ended_by_substitution": bool(ended_by_substitution),
    })

In [39]:
# Cell 2: build first fact_lineup_stint prototype from grouped timestamps

# Assumes you already have:
# - pbp_df
# - event_groups_df
# - sub_group_lookup
# - apply_substitution_batch(...)
# - home_team_id, away_team_id
# - current_home_lineup, current_away_lineup

# 1) Ensure event group order = game order
eg = event_groups_df.copy().sort_values(
    ["period", "clock_seconds_remaining"],
    ascending=[True, False],  # within period, clock counts down
    kind="mergesort"
).reset_index(drop=True)

# 2) State
home_lineup = set(current_home_lineup)
away_lineup = set(current_away_lineup)

stint_rows = []

# Open first stint at first timestamp
first = eg.iloc[0]
cur_period = int(first["period"])
cur_start_clock = first["clock"]
cur_start_sec = float(first["clock_seconds_remaining"])
cur_score_home_start, cur_score_away_start = get_score_snapshot_for_group(
    pbp_df, cur_period, cur_start_clock
)

for i, row in eg.iterrows():
    period = int(row["period"])
    clock = row["clock"]
    sec = float(row["clock_seconds_remaining"])
    has_sub = bool(row["has_substitution"])

    # Score snapshot at this timestamp (used when closing stints)
    score_home_here, score_away_here = get_score_snapshot_for_group(pbp_df, period, clock)

    # A) Period changed -> close previous period at 00:00 once
    if period != cur_period:
        append_stint_row(
            out_rows=stint_rows,
            period=cur_period,
            start_clock=cur_start_clock,
            end_clock="PT00M00.00S",
            start_sec=cur_start_sec,
            end_sec=0.0,
            home_lineup=home_lineup,
            away_lineup=away_lineup,
            score_home_start=cur_score_home_start,
            score_away_start=cur_score_away_start,
            score_home_end=score_home_here,
            score_away_end=score_away_here,
            ended_by_substitution=False,
        )

        # Open new stint at start of current period timestamp with same lineups
        cur_period = period
        cur_start_clock = clock
        cur_start_sec = sec
        cur_score_home_start, cur_score_away_start = score_home_here, score_away_here

    # B) If this timestamp has substitution(s), close once, apply batch once, reopen once
    if has_sub:
        # close old stint at this timestamp boundary
        append_stint_row(
            out_rows=stint_rows,
            period=cur_period,
            start_clock=cur_start_clock,
            end_clock=clock,
            start_sec=cur_start_sec,
            end_sec=sec,
            home_lineup=home_lineup,
            away_lineup=away_lineup,
            score_home_start=cur_score_home_start,
            score_away_start=cur_score_away_start,
            score_home_end=score_home_here,
            score_away_end=score_away_here,
            ended_by_substitution=True,
        )

        # apply full batch at this exact timestamp
        batch = sub_group_lookup.get((period, clock), [])
        home_lineup, away_lineup = apply_substitution_batch(
            home_lineup, away_lineup, batch, home_team_id, away_team_id
        )

        # sanity: keep 5 players each
        if len(home_lineup) != 5:
            print(f"[WARN] HOME lineup size != 5 at ({period}, {clock}): {len(home_lineup)}")
        if len(away_lineup) != 5:
            print(f"[WARN] AWAY lineup size != 5 at ({period}, {clock}): {len(away_lineup)}")

        # start ONE new stint at same timestamp (no append here)
        cur_start_clock = clock
        cur_start_sec = sec
        cur_score_home_start, cur_score_away_start = score_home_here, score_away_here

# 3) Final flush: close open stint at end of last period
last_period = int(eg.iloc[-1]["period"])
last_score_home, last_score_away = get_score_snapshot_for_group(
    pbp_df,
    int(eg.iloc[-1]["period"]),
    eg.iloc[-1]["clock"]
)

append_stint_row(
    out_rows=stint_rows,
    period=cur_period,
    start_clock=cur_start_clock,
    end_clock="PT00M00.00S",
    start_sec=cur_start_sec,
    end_sec=0.0,
    home_lineup=home_lineup,
    away_lineup=away_lineup,
    score_home_start=cur_score_home_start,
    score_away_start=cur_score_away_start,
    score_home_end=last_score_home,
    score_away_end=last_score_away,
    ended_by_substitution=False,
)

fact_lineup_stint = pd.DataFrame(stint_rows)
print("Built fact_lineup_stint rows:", len(fact_lineup_stint))

[WARN] HOME outgoing player not found: Garza
[WARN] HOME outgoing player not found: Scheierman
[WARN] HOME lineup size != 5 at (2, PT08M43.00S): 6
[WARN] AWAY lineup size != 5 at (2, PT08M43.00S): 4
[WARN] HOME outgoing player not found: Hauser
[WARN] AWAY outgoing player not found: Drummond
[WARN] HOME lineup size != 5 at (2, PT07M16.00S): 6
[WARN] HOME lineup size != 5 at (2, PT06M50.00S): 6
[WARN] AWAY lineup size != 5 at (2, PT06M50.00S): 4
[WARN] HOME lineup size != 5 at (2, PT05M04.00S): 6
[WARN] AWAY lineup size != 5 at (2, PT05M04.00S): 4
[WARN] HOME lineup size != 5 at (2, PT04M39.00S): 6
[WARN] AWAY lineup size != 5 at (2, PT04M39.00S): 4
[WARN] HOME lineup size != 5 at (2, PT02M47.00S): 6
[WARN] AWAY lineup size != 5 at (2, PT02M47.00S): 4
[WARN] HOME lineup size != 5 at (2, PT02M10.00S): 6
[WARN] AWAY lineup size != 5 at (2, PT02M10.00S): 4
[WARN] HOME lineup size != 5 at (2, PT01M50.00S): 6
[WARN] AWAY lineup size != 5 at (2, PT01M50.00S): 4
[WARN] HOME lineup size != 5 at

In [40]:
# Cell 1: Manually seed period-start lineups (NAMES, one game)

home_team_id = 1610612738  # example BOS
away_team_id = 1610612755  # example PHI

# Define lineups you trust at each period start.
# Use period numbers as keys (1..4, and 5+ for OT if needed).
period_start_lineups = {
    1: {
        "home": {"Hauser", "Tatum", "Queta", "Brown", "White"},
        "away": {"George", "Oubre Jr.", "Bona", "Edgecombe", "Maxey"},
    },
    2: {
        "home": {"Pritchard", "Walsh", "Brown", "White", "Scheierman"},
        "away": {"George", "Grimes", "Barlow", "Edwards", "Drummond"},
    },
    3: {
        "home": {"Hauser", "Tatum", "Vucevic", "Brown", "White"},
        "away": {"George", "Oubre Jr.", "Bona", "Edgecombe", "Maxey"},
    },
    4: {
        "home": {"Hauser", "Tatum", "Vucevic", "Brown", "White"},
        "away": {"George", "Oubre Jr.", "Drummond", "Edgecombe", "Maxey"},
    },
}

# Quick sanity check
for p, d in period_start_lineups.items():
    assert len(d["home"]) == 5, f"Period {p} home lineup != 5"
    assert len(d["away"]) == 5, f"Period {p} away lineup != 5"

print("Period-start lineup seeds loaded:")
for p in sorted(period_start_lineups):
    print(f"  P{p} HOME={sorted(period_start_lineups[p]['home'])}")
    print(f"      AWAY={sorted(period_start_lineups[p]['away'])}")

Period-start lineup seeds loaded:
  P1 HOME=['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
      AWAY=['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
  P2 HOME=['Brown', 'Pritchard', 'Scheierman', 'Walsh', 'White']
      AWAY=['Barlow', 'Drummond', 'Edwards', 'George', 'Grimes']
  P3 HOME=['Brown', 'Hauser', 'Tatum', 'Vucevic', 'White']
      AWAY=['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']
  P4 HOME=['Brown', 'Hauser', 'Tatum', 'Vucevic', 'White']
      AWAY=['Drummond', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']


In [41]:
# Cell 2: Helper to reset lineup state at period start

def reset_lineup_state_for_period(period, period_start_lineups):
    if period not in period_start_lineups:
        raise ValueError(f"No manual lineup seed found for period {period}")

    home = set(period_start_lineups[period]["home"])
    away = set(period_start_lineups[period]["away"])

    if len(home) != 5 or len(away) != 5:
        raise ValueError(f"Period {period} lineup seed must be 5 players each")

    print(f"\n[RESET] Period {period} lineup state")
    print(f"  HOME: {sorted(home)}")
    print(f"  AWAY: {sorted(away)}")

    return home, away

In [42]:
# Cell 3: Replay substitution batches period-by-period with reset logic

# Assumes:
# - event_groups_df has one row per (period, clock) and includes has_substitution
# - sub_group_lookup maps (period, clock) -> list of sub dicts
# - apply_substitution_batch(...) already exists and works with names

eg = event_groups_df.copy().sort_values(
    ["period", "clock_seconds_remaining"],
    ascending=[True, False],   # time remaining desc inside period
    kind="mergesort"
).reset_index(drop=True)

current_period = None
current_home_lineup = None
current_away_lineup = None

validation_rows = []

for _, row in eg.iterrows():
    period = int(row["period"])
    clock = row["clock"]
    has_sub = bool(row["has_substitution"])

    # Reset lineup state at period transition
    if current_period != period:
        current_period = period
        current_home_lineup, current_away_lineup = reset_lineup_state_for_period(
            period, period_start_lineups
        )

    # Process substitution batch only when present
    if has_sub:
        batch = sub_group_lookup.get((period, clock), [])
        print(f"\n[P{period} {clock}] applying batch with {len(batch)} substitutions")

        before_home = set(current_home_lineup)
        before_away = set(current_away_lineup)

        current_home_lineup, current_away_lineup = apply_substitution_batch(
            current_home_lineup,
            current_away_lineup,
            batch,
            home_team_id,
            away_team_id,
        )

        # Validation checks after each batch
        home_ok = len(current_home_lineup) == 5
        away_ok = len(current_away_lineup) == 5

        if not home_ok or not away_ok:
            print("[ERROR] lineup size drift detected")
            print("  HOME size:", len(current_home_lineup), sorted(current_home_lineup))
            print("  AWAY size:", len(current_away_lineup), sorted(current_away_lineup))
        else:
            print("  HOME:", sorted(current_home_lineup))
            print("  AWAY:", sorted(current_away_lineup))

        validation_rows.append({
            "period": period,
            "clock": clock,
            "n_subs": len(batch),
            "home_size_ok": home_ok,
            "away_size_ok": away_ok,
            "home_lineup_before": sorted(before_home),
            "away_lineup_before": sorted(before_away),
            "home_lineup_after": sorted(current_home_lineup),
            "away_lineup_after": sorted(current_away_lineup),
        })

validation_df = pd.DataFrame(validation_rows)
print("\nReplay complete.")
print("Batches processed:", len(validation_df))
print("Any home size failures:", (~validation_df["home_size_ok"]).any() if len(validation_df) else False)
print("Any away size failures:", (~validation_df["away_size_ok"]).any() if len(validation_df) else False)


[RESET] Period 1 lineup state
  HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
  AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']

[P1 PT10M23.00S] applying batch with 1 substitutions
  HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
  AWAY: ['Drummond', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']

[P1 PT08M39.00S] applying batch with 1 substitutions
  HOME: ['Brown', 'Hauser', 'Tatum', 'Vucevic', 'White']
  AWAY: ['Drummond', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']

[P1 PT04M50.00S] applying batch with 3 substitutions
  HOME: ['Brown', 'Pritchard', 'Tatum', 'Vucevic', 'White']
  AWAY: ['Barlow', 'Drummond', 'George', 'Grimes', 'Maxey']

[P1 PT04M39.00S] applying batch with 1 substitutions
  HOME: ['Brown', 'Pritchard', 'Tatum', 'Vucevic', 'White']
  AWAY: ['Barlow', 'George', 'Grimes', 'Maxey', 'Oubre Jr.']

[P1 PT03M49.00S] applying batch with 1 substitutions
  HOME: ['Pritchard', 'Tatum', 'Vucevic', 'Walsh', 'White']
  AWAY: ['Barlow', 'George', 'Grimes'

In [43]:
# Cell 4: Inspect validation summary

if len(validation_df) == 0:
    print("No substitution batches found in replay window.")
else:
    display(validation_df.head(20))

    failures = validation_df[(~validation_df["home_size_ok"]) | (~validation_df["away_size_ok"])]
    print("Failure rows:", len(failures))
    if len(failures):
        display(failures[["period", "clock", "n_subs", "home_size_ok", "away_size_ok"]])

,period,clock,n_subs,home_size_ok,away_size_ok,home_lineup_before,away_lineup_before,home_lineup_after,away_lineup_after
0,1,PT10M23.00S,1,True,True,"[Brown, Hauser, Queta, Tatum, White]","[Bona, Edgecombe, George, Maxey, Oubre Jr.]","[Brown, Hauser, Queta, Tatum, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]"
1,1,PT08M39.00S,1,True,True,"[Brown, Hauser, Queta, Tatum, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]","[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]"
2,1,PT04M50.00S,3,True,True,"[Brown, Hauser, Tatum, Vucevic, White]","[Drummond, Edgecombe, George, Maxey, Oubre Jr.]","[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, Drummond, George, Grimes, Maxey]"
3,1,PT04M39.00S,1,True,True,"[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, Drummond, George, Grimes, Maxey]","[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]"
4,1,PT03M49.00S,1,True,True,"[Brown, Pritchard, Tatum, Vucevic, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]","[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]"
5,1,PT02M39.00S,1,True,True,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, George, Grimes, Maxey, Oubre Jr.]","[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]"
6,1,PT00M27.30S,2,True,True,"[Pritchard, Tatum, Vucevic, Walsh, White]","[Barlow, Edwards, Grimes, Maxey, Oubre Jr.]","[Brown, Pritchard, Tatum, Walsh, White]","[Edwards, George, Grimes, Maxey, Oubre Jr.]"
7,2,PT08M43.00S,3,True,True,"[Brown, Pritchard, Scheierman, Walsh, White]","[Barlow, Drummond, Edwards, George, Grimes]","[Brown, Pritchard, Queta, Walsh, White]","[Barlow, Drummond, Edwards, George, Maxey]"
8,2,PT07M16.00S,2,False,True,"[Brown, Pritchard, Queta, Walsh, White]","[Barlow, Drummond, Edwards, George, Maxey]","[Brown, Pritchard, Queta, Tatum, Walsh, White]","[Barlow, Bona, Edwards, George, Maxey]"
9,2,PT06M50.00S,1,False,True,"[Brown, Pritchard, Queta, Tatum, Walsh, White]","[Barlow, Bona, Edwards, George, Maxey]","[Brown, Pritchard, Queta, Tatum, Walsh, White]","[Barlow, Bona, Edwards, Maxey, Oubre Jr.]"


Failure rows: 12


,period,clock,n_subs,home_size_ok,away_size_ok
8,2,PT07M16.00S,2,False,True
9,2,PT06M50.00S,1,False,True
10,2,PT05M04.00S,1,False,True
11,2,PT04M39.00S,1,False,True
12,2,PT02M47.00S,1,False,True
13,2,PT02M10.00S,1,False,True
14,2,PT01M50.00S,1,False,True
15,2,PT01M40.00S,1,False,True
16,2,PT01M14.00S,1,False,True
28,4,PT05M52.00S,1,False,True


In [ ]:
# Cell 1: Strict substitution batch applier with rich debug output

def apply_substitution_batch_strict(
    home_lineup: set[str],
    away_lineup: set[str],
    sub_batch: list[dict],
    home_team_id: int,
    away_team_id: int,
):
    """
    Apply one timestamp's substitution batch with strict checks.

    Rules:
      - outgoing must already be on floor
      - incoming must NOT already be on floor

    Returns:
      new_home_lineup, new_away_lineup, sub_debug_rows (list of dict)
    """
    home = set(home_lineup)
    away = set(away_lineup)
    debug_rows = []

    for sub in sub_batch:
        team_id = int(sub["teamId"])
        player_in = str(sub["player_in_name"]).strip()
        # Prefer API outgoing name; fallback keeps old parsed objects from breaking reruns.
        player_out = str(
            sub.get("player_out_name_api")
            or sub.get("player_out_name")
            or sub.get("player_out_name_text")
            or ""
        ).strip()

        if team_id == home_team_id:
            team_label = "HOME"
            lineup = home
        elif team_id == away_team_id:
            team_label = "AWAY"
            lineup = away
        else:
            debug_rows.append({
                "team": "UNKNOWN",
                "team_id": team_id,
                "player_out": player_out,
                "player_in": player_in,
                "outgoing_found": False,
                "incoming_already_present": False,
                "sub_succeeded": False,
                "error_type": "unknown_team",
                "message": f"Unknown teamId={team_id}",
            })
            continue

        outgoing_found = player_out in lineup
        incoming_already_present = player_in in lineup

        sub_succeeded = outgoing_found and (not incoming_already_present)
        error_type = None
        msg = "ok"

        if not outgoing_found and incoming_already_present:
            error_type = "out_missing_and_in_already_present"
            msg = f"{team_label}: OUT '{player_out}' missing, IN '{player_in}' already on floor"
        elif not outgoing_found:
            error_type = "outgoing_not_found"
            msg = f"{team_label}: OUT '{player_out}' not found on floor"
        elif incoming_already_present:
            error_type = "incoming_already_present"
            msg = f"{team_label}: IN '{player_in}' already on floor"

        if sub_succeeded:
            lineup.remove(player_out)
            lineup.add(player_in)
        else:
            print(f"[WARN] {msg}")

        debug_rows.append({
            "team": team_label,
            "team_id": team_id,
            "player_out": player_out,
            "player_in": player_in,
            "outgoing_found": outgoing_found,
            "incoming_already_present": incoming_already_present,
            "sub_succeeded": sub_succeeded,
            "error_type": error_type,
            "message": msg,
            "description": sub.get("description"),
            "actionNumber": sub.get("actionNumber"),
        })

    return home, away, debug_rows

In [ ]:
# Cell 2: Replay all substitution batches with detailed logging dataframe

import pandas as pd

# Assumes already defined:
# - event_groups_df
# - sub_group_lookup
# - period_start_lineups
# - home_team_id, away_team_id

def reset_lineup_state_for_period(period, period_start_lineups):
    if period not in period_start_lineups:
        raise ValueError(f"No lineup seed for period {period}")
    h = set(period_start_lineups[period]["home"])
    a = set(period_start_lineups[period]["away"])
    if len(h) != 5 or len(a) != 5:
        raise ValueError(f"Period {period} seed must have 5 players each")
    print(f"\n[RESET] P{period}")
    print(" HOME:", sorted(h))
    print(" AWAY:", sorted(a))
    return h, a

eg = event_groups_df.copy().sort_values(
    ["period", "clock_seconds_remaining"],
    ascending=[True, False],
    kind="mergesort"
).reset_index(drop=True)

current_period = None
home_state, away_state = None, None
validation_rows = []

for _, row in eg.iterrows():
    period = int(row["period"])
    clock = row["clock"]
    has_sub = bool(row["has_substitution"])

    if period != current_period:
        current_period = period
        home_state, away_state = reset_lineup_state_for_period(period, period_start_lineups)

    if not has_sub:
        continue

    sub_batch = sub_group_lookup.get((period, clock), [])
    home_before = sorted(home_state)
    away_before = sorted(away_state)

    home_state, away_state, sub_debug = apply_substitution_batch_strict(
        home_state,
        away_state,
        sub_batch,
        home_team_id,
        away_team_id,
    )

    home_ok = len(home_state) == 5
    away_ok = len(away_state) == 5

    for d in sub_debug:
        validation_rows.append({
            "period": period,
            "clock": clock,
            "team": d["team"],
            "team_id": d["team_id"],
            "player_out": d["player_out"],
            "player_in": d["player_in"],
            "outgoing_found": d["outgoing_found"],
            "incoming_already_present": d["incoming_already_present"],
            "sub_succeeded": d["sub_succeeded"],
            "error_type": d["error_type"],
            "message": d["message"],
            "home_size_ok": home_ok,
            "away_size_ok": away_ok,
            "home_lineup_before": home_before,
            "away_lineup_before": away_before,
            "home_lineup_after": sorted(home_state),
            "away_lineup_after": sorted(away_state),
            "description": d.get("description"),
            "actionNumber": d.get("actionNumber"),
        })

validation_df = pd.DataFrame(validation_rows)

print("\nReplay complete.")
print("Sub rows logged:", len(validation_df))
print("Sub failures:", int((~validation_df["sub_succeeded"]).sum()) if len(validation_df) else 0)
print("Home size failures:", int((~validation_df["home_size_ok"]).sum()) if len(validation_df) else 0)
print("Away size failures:", int((~validation_df["away_size_ok"]).sum()) if len(validation_df) else 0)

display(validation_df.head(30))


[RESET] P1
 HOME: ['Brown', 'Hauser', 'Queta', 'Tatum', 'White']
 AWAY: ['Bona', 'Edgecombe', 'George', 'Maxey', 'Oubre Jr.']


KeyError: 'player_out_name'